In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import scipy

### Data Loader Helpers

In [ ]:
CATEGORIES = ['BIRD.npz', 'PROGRAMMING_LANGUAGE.npz', 'DRUG.npz', 'ACTOR.npz', 'LAKE.npz', 'STAR.npz', 'BOOK.npz', 'COUNTRY.npz', 'MUSHROOM.npz', 'CAR.npz', 'PHONE.npz', 'REPTILE.npz', 'THEORY.npz', 'SPORT.npz', 'MINERAL.npz', 'SCIENTIST.npz', 'MOUNTAIN.npz', 'TUNNEL.npz', 'TREE.npz', 'BRIDGE.npz', 'SONG.npz', 'POLITICIAN.npz', 'MAMMAL.npz', 'AIRPORT.npz', 'DISEASE.npz', 'CITY.npz', 'ATHLETE.npz', 'MOVIE.npz', 'COMPANY.npz', 'AWARD.npz', 'GALAXY.npz', 'ARTIST.npz', 'WRITER.npz', 'PLANT.npz', 'LIBRARY_PYTHON.npz', 'SPORTS_TEAM.npz', 'MUSEUM.npz', 'COMPOUND.npz', 'ISLAND.npz', 'PAINTING.npz', 'RIVER.npz', 'AMPHIBIAN.npz', 'LIBRARY_JAVASCRIPT.npz']

DATA_PATH_OLMO = 'data_olmo'
DATA_PATH_DEEPSEEK = 'data_deepseek'
DATA_PATH_LLAMA = 'data_llama'

In [ ]:
def load_data_olmo(question_type, location='LAY', idx_layer=39):
    Xs = []
    ys = []
    data_categories = []

    for idx_category, category in enumerate(CATEGORIES):
        cur_data = np.load(os.path.join(DATA_PATH_OLMO, category))
        cur_data_real = cur_data[f'activations_REAL_{question_type}_{location}'][idx_layer]
        cur_data_fabr = cur_data[f'activations_FABR_{question_type}_{location}'][idx_layer]

        Xs.append(cur_data_real)
        ys.append(np.zeros(cur_data_real.shape[0]))
        Xs.append(cur_data_fabr)
        ys.append(np.ones(cur_data_fabr.shape[0]))

        data_categories.append(np.full(cur_data_real.shape[0]+cur_data_fabr.shape[0], idx_category))

    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(data_categories)

In [ ]:
def load_data_deepseek(question_type):
    Xs = []
    ys = []
    data_categories = []

    for idx_category, category in enumerate(CATEGORIES):
        cur_data = np.load(os.path.join(DATA_PATH_DEEPSEEK, category))
        cur_data_real = cur_data[f'activations_REAL_{question_type}_LAY']
        cur_data_fabr = cur_data[f'activations_FABR_{question_type}_LAY']

        Xs.append(cur_data_real)
        ys.append(np.zeros(cur_data_real.shape[0]))
        Xs.append(cur_data_fabr)
        ys.append(np.ones(cur_data_fabr.shape[0]))

        data_categories.append(np.full(cur_data_real.shape[0]+cur_data_fabr.shape[0], idx_category))

    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(data_categories)

In [ ]:
def load_data_llama(question_type):
    Xs = []
    ys = []
    data_categories = []

    for idx_category, category in enumerate(CATEGORIES):
        cur_data = np.load(os.path.join(DATA_PATH_LLAMA, category))
        cur_data_real = cur_data[f'activations_REAL_{question_type}_LAY']
        cur_data_fabr = cur_data[f'activations_FABR_{question_type}_LAY']

        Xs.append(cur_data_real)
        ys.append(np.zeros(cur_data_real.shape[0]))
        Xs.append(cur_data_fabr)
        ys.append(np.ones(cur_data_fabr.shape[0]))

        data_categories.append(np.full(cur_data_real.shape[0]+cur_data_fabr.shape[0], idx_category))

    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(data_categories)

In [ ]:
def load_results_olmo(question_type, location='LAY', idx_layer=39, preprocessing='id', classifier='SVM'):
    return np.load(f'results/{question_type}_{location}_{idx_layer}_{preprocessing}_{classifier}.npz')

def load_results_deepseek(question_type):
    return np.load(f'results_deepseek/{question_type}_LAY_39_id_SVM.npz')

def load_results_llama(question_type):
    return np.load(f'results_llama/{question_type}_LAY_39_id_SVM.npz')

# Test

In [ ]:
def load_data(llm):
    if llm == 'Olmo 2':
        probas_prop = load_results_olmo('PROP')['probas']
        probas_ex = load_results_olmo('EX')['probas']
        _, y_prop, _ = load_data_olmo('PROP')
        _, y_ex, _ = load_data_olmo('EX')
        experiment = '\\modelOlmo'
    elif llm == 'Deepseek-r1':
        probas_prop = load_results_deepseek('PROP')['probas']
        probas_ex = load_results_deepseek('EX')['probas']
        _, y_prop, _ = load_data_deepseek('PROP')
        _, y_ex, _ = load_data_deepseek('EX')
        experiment = '\\modelDeepseek'
    elif llm == 'Llama 3.3':
        probas_prop = load_results_llama('PROP')['probas']
        probas_ex = load_results_llama('EX')['probas']
        _, y_prop, _ = load_data_llama('PROP')
        _, y_ex, _ = load_data_llama('EX')
        experiment = '\\modelLlama'

    return probas_ex, y_ex, probas_prop, y_prop

In [ ]:
def roc_difference_test(model, p, interval_start=-3, interval_stop=3, interval_step=0.01, plots=False):
    interval = np.arange(interval_start, interval_stop, interval_step)

    # Load Data
    probas_ex, y_ex, probas_prop, y_prop = load_data(model)

    # Split data into positives and negatives
    probas_ex_real = probas_ex[y_ex == 0]
    probas_ex_fabr = probas_ex[y_ex == 1]

    probas_prop_real = probas_prop[y_prop == 0]
    probas_prop_fabr = probas_prop[y_prop == 1]

    # Get sample sizes
    m = probas_ex_real.shape[0]
    n = probas_prop_real.shape[0]

    # Create CDFs for each distribution
    fn = scipy.stats.ecdf(probas_ex_real).cdf
    fp = scipy.stats.ecdf(probas_ex_fabr).cdf

    gn = scipy.stats.ecdf(probas_prop_real).cdf
    gp = scipy.stats.ecdf(probas_prop_fabr).cdf

    # Mixture of distributions (equal classes)
    def f_forward(x):
        return .5*fn.evaluate(x) + .5*fp.evaluate(x)

    def g_forward(x):
        return .5*gn.evaluate(x) + .5*gp.evaluate(x)
    
    # Calculate interval mapping, tau
    target_p = f_forward(interval)
    cur_gs = g_forward(interval)[:,np.newaxis] < target_p
    tau_vals = np.concat([interval, [interval_stop]])[np.sum(cur_gs, axis=0)]

    def tau(x):
        idx = np.sum(interval[:,np.newaxis] < x, axis=0)
        return tau_vals[idx]
    
    # Calculate KS-Test statistics
    dn = np.max(np.abs(fn.evaluate(interval) - gn.evaluate(tau(interval))))
    dp = np.max(np.abs(fp.evaluate(interval) - gp.evaluate(tau(interval))))

    # Calculate p-value
    success, dn, pn = scipy.stats._stats_py._attempt_exact_2kssamp(n, m, np.gcd(n, m), dn, 'two-sided')
    if not success:
        print('Could not compute exact distribution.')
        return
    success, dp, pp = scipy.stats._stats_py._attempt_exact_2kssamp(n, m, np.gcd(n, m), dp, 'two-sided')
    if not success:
        print('Could not compute exact distribution.')
        return

    # Bonferroni correction of p-threshold
    p_corrected = p/2

    # Print results
    print(f'Statistic for negative class: {dn} ({"significant" if pn < p_corrected else "not significant"}) (pvalue: {pn})')
    print(f'Statistic for positive class: {dp} ({"significant" if pp < p_corrected else "not significant"}) (pvalue: {pp})')

    if plots:
        plt.figure()
        plt.title('Positive class')
        plt.plot(interval, fp.evaluate(interval), label='F (ex)')
        plt.plot(interval, gp.evaluate(tau(interval)), label='G (prop)')
        plt.xlabel('Score')
        plt.ylabel('Cumulative Density')
        plt.legend()
        plt.show()


        plt.figure()
        plt.title('Negative class')
        plt.plot(interval, fn.evaluate(interval), label='F (ex)')
        plt.plot(interval, gn.evaluate(tau(interval)), label='G (prop)')
        plt.xlabel('Score')
        plt.ylabel('Cumulative Density')
        plt.legend()
        plt.show()

In [ ]:
roc_difference_test('Olmo 2', 0.05)

In [ ]:
roc_difference_test('Deepseek-r1', 0.05)

In [ ]:
roc_difference_test('Llama 3.3', 0.05)